PHASE : 1A

In [ ]:
"""
═══════════════════════════════════════════════════════════════════
AETHER — Phase 1A: Text Emotion Model Training Script
═══════════════════════════════════════════════════════════════════

Fine-tunes DistilRoBERTa on emotion classification with 15 core
Aether emotion categories + intensity levels.

15 Emotions:
  happy, sad, angry, calm, anxious, energetic, focused,
  nostalgic, romantic, melancholic, confident, hopeful,
  frustrated, lonely, dreamy

Datasets (2025 latest):
  - Primary: 20-Emotion Text Classification Dataset (HuggingFace)
  - Fallback: GoEmotions (Google)

Designed for:
  - Google Colab T4 GPU (recommended for training)
  - MacBook Air M2 (MPS backend)
  - CPU (slower but works)

Usage:
  Colab:  Upload → Run each section in order
  Local:  python phase_1a_text_emotion/train_text_emotion.py
═══════════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════
# SECTION 1: Install & Import
# ═══════════════════════════════════════════════
# Colab first cell:
# !pip install -q transformers datasets accelerate scikit-learn torch evaluate

import os
import sys
import json
import warnings
import numpy as np
import torch
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False


# Detect environment
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("⚠️  CPU mode — training will be slower")

print(f"   PyTorch {torch.__version__} | Device: {DEVICE}")


# ═══════════════════════════════════════════════
# SECTION 2: Aether 15 Emotion System
# ═══════════════════════════════════════════════

AETHER_EMOTIONS = [
    "happy", "sad", "angry", "calm", "anxious",
    "energetic", "focused", "nostalgic", "romantic",
    "melancholic", "confident", "hopeful", "frustrated",
    "lonely", "dreamy",
]
NUM_LABELS = len(AETHER_EMOTIONS)  # 15

EMOTION_TO_ID = {e: i for i, e in enumerate(AETHER_EMOTIONS)}
ID_TO_EMOTION = {i: e for i, e in enumerate(AETHER_EMOTIONS)}

# 150+ fine-grained labels → 15 core emotions
# pride → confident (not happy), excitement → energetic (not happy)
FINE_TO_CORE = {
    # ─── happy ───
    "joy": "happy", "happiness": "happy", "amusement": "happy",
    "gratitude": "happy",
    "admiration": "happy", "approval": "happy",
    "relief": "happy", "happy": "happy", "elation": "happy",
    "contentment": "happy", "enthusiasm": "happy", "delight": "happy",
    "cheerfulness": "happy", "satisfaction": "happy", "euphoria": "happy",
    "bliss": "happy", "pleasure": "happy", "ecstasy": "happy",
    "glee": "happy", "jubilation": "happy", "exuberance": "happy",

    # ─── sad ───
    "sadness": "sad", "sad": "sad", "grief": "sad",
    "disappointment": "sad", "remorse": "sad", "sorrow": "sad",
    "despair": "sad", "guilt": "sad", "regret": "sad",
    "heartbreak": "sad", "misery": "sad", "anguish": "sad",
    "dejection": "sad", "gloom": "sad", "unhappiness": "sad",
    "hurt": "sad", "suffering": "sad",

    # ─── angry ───
    "anger": "angry", "angry": "angry", "rage": "angry",
    "hate": "angry", "hostility": "angry", "outrage": "angry",
    "fury": "angry", "wrath": "angry", "contempt": "angry",
    "resentment": "angry", "bitterness": "angry", "aggression": "angry",
    "indignation": "angry", "vengeance": "angry", "disgust": "angry",

    # ─── calm ───
    "calm": "calm", "serenity": "calm", "peace": "calm",
    "relaxation": "calm", "tranquility": "calm", "composure": "calm",
    "patience": "calm", "stillness": "calm", "harmony": "calm",
    "balance": "calm", "acceptance": "calm", "equanimity": "calm",
    "neutral": "calm", "indifference": "calm",

    # ─── anxious ───
    "fear": "anxious", "anxious": "anxious", "anxiety": "anxious",
    "nervousness": "anxious", "worry": "anxious", "panic": "anxious",
    "apprehension": "anxious", "dread": "anxious", "unease": "anxious",
    "tension": "anxious", "distress": "anxious", "alarm": "anxious",
    "trepidation": "anxious", "paranoia": "anxious",
    "embarrassment": "anxious", "shame": "anxious",

    # ─── energetic ───
    "energetic": "energetic", "excitement": "energetic", "thrill": "energetic",
    "exhilaration": "energetic", "adrenaline": "energetic",
    "vigor": "energetic", "vitality": "energetic",
    "zeal": "energetic", "dynamism": "energetic",
    "liveliness": "energetic", "spirited": "energetic",

    # ─── focused ───
    "focused": "focused", "concentration": "focused",
    "determination": "focused", "attentiveness": "focused",
    "resolve": "focused", "dedication": "focused",
    "mindfulness": "focused", "absorption": "focused",
    "engagement": "focused", "vigilance": "focused",
    "deliberation": "focused", "interest": "focused",

    # ─── nostalgic ───
    "nostalgia": "nostalgic", "nostalgic": "nostalgic",
    "reminiscence": "nostalgic", "wistfulness": "nostalgic",
    "sentimentality": "nostalgic", "yearning": "nostalgic",
    "homesickness": "nostalgic", "bittersweet": "nostalgic",
    "remembrance": "nostalgic",

    # ─── romantic ───
    "romantic": "romantic", "romance": "romantic",
    "affection": "romantic", "tenderness": "romantic",
    "intimacy": "romantic", "passion": "romantic",
    "devotion": "romantic", "adoration": "romantic",
    "infatuation": "romantic", "desire": "romantic",
    "longing": "romantic", "warmth": "romantic",
    "caring": "romantic", "fondness": "romantic", "love": "romantic",

    # ─── melancholic ───
    "melancholy": "melancholic", "melancholic": "melancholic",
    "pensiveness": "melancholic", "gloominess": "melancholic",
    "existential": "melancholic", "somber": "melancholic",
    "mournful": "melancholic", "brooding": "melancholic",
    "dismal": "melancholic", "morose": "melancholic",
    "heavyhearted": "melancholic", "desolation": "melancholic",

    # ─── confident ───
    "confidence": "confident", "confident": "confident", "pride": "confident",
    "empowerment": "confident", "boldness": "confident",
    "courage": "confident", "assertiveness": "confident",
    "strength": "confident", "power": "confident",
    "dominance": "confident", "triumph": "confident",
    "victory": "confident", "invincibility": "confident",

    # ─── hopeful ───
    "hope": "hopeful", "hopeful": "hopeful",
    "optimism": "hopeful", "anticipation": "hopeful",
    "aspiration": "hopeful", "faith": "hopeful",
    "expectation": "hopeful", "encouragement": "hopeful",
    "inspiration": "hopeful", "possibility": "hopeful",

    # ─── frustrated ───
    "frustration": "frustrated", "frustrated": "frustrated", "jealousy": "frustrated",
    "annoyance": "frustrated", "irritation": "frustrated",
    "exasperation": "frustrated", "impatience": "frustrated",
    "vexation": "frustrated", "agitation": "frustrated",
    "disapproval": "frustrated", "displeasure": "frustrated",
    "dissatisfaction": "frustrated", "aggravation": "frustrated",

    # ─── lonely ───
    "loneliness": "lonely", "lonely": "lonely",
    "isolation": "lonely", "solitude": "lonely",
    "abandonment": "lonely", "alienation": "lonely",
    "emptiness": "lonely", "disconnection": "lonely",
    "forsaken": "lonely", "excluded": "lonely",

    # ─── dreamy ───
    "dreamy": "dreamy", "wonder": "dreamy",
    "imagination": "dreamy", "fantasy": "dreamy",
    "ethereal": "dreamy", "surreal": "dreamy",
    "whimsical": "dreamy", "mystical": "dreamy",
    "enchantment": "dreamy", "reverie": "dreamy",
    "daydreaming": "dreamy", "trance": "dreamy",
    "curiosity": "dreamy", "awe": "dreamy",
    "fascination": "dreamy", "amazement": "dreamy",
    "surprise": "dreamy", "confusion": "dreamy",
    "realization": "dreamy", "mysterious": "dreamy",
}


def map_label_to_aether(label: str) -> str | None:
    """Map any fine-grained emotion label to one of 15 Aether core emotions."""
    label_lower = label.lower().strip().replace(" ", "_").replace("-", "_")
    return FINE_TO_CORE.get(label_lower, None)


# Model & training config
MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 32 if DEVICE == "cuda" else 16
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

OUTPUT_DIR = Path("./models/text_emotion")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📋 Aether Text Emotion Model Config:")
print(f"   Model: {MODEL_NAME}")
print(f"   Emotions: {NUM_LABELS} categories")
print(f"   Categories: {AETHER_EMOTIONS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Output: {OUTPUT_DIR}")


# ═══════════════════════════════════════════════
# SECTION 3: Load BOTH Datasets
# ═══════════════════════════════════════════════
# Strategy: Combine 20-Emotion + GoEmotions for
# maximum real-data coverage (12/15 emotions),
# then augment 3 remaining gaps with curated
# seed examples — standard data augmentation.

from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets

print("\n📥 Loading datasets (combining two sources for max coverage)...")

TEXT_COLUMN = "text"
datasets_loaded = []

# ── Dataset 1: 20-Emotion (2025) ──
try:
    ds_20 = load_dataset("shreyaspulle98/emotion-dataset-20-emotions")
    print(f"✅ Loaded: 20-Emotion (2025)")
    for split in ds_20:
        print(f"   {split}: {len(ds_20[split]):,} samples")
    datasets_loaded.append(("20-Emotion", ds_20))
except Exception as e:
    print(f"⚠️  20-Emotion not available: {e}")

# ── Dataset 2: GoEmotions (Google) ──
try:
    ds_go = load_dataset("google-research-datasets/go_emotions", "simplified")
    print(f"✅ Loaded: GoEmotions (Google)")
    for split in ds_go:
        print(f"   {split}: {len(ds_go[split]):,} samples")
    datasets_loaded.append(("GoEmotions", ds_go))
except Exception as e:
    print(f"⚠️  GoEmotions not available: {e}")

if not datasets_loaded:
    print("❌ Could not load any dataset!")
    sys.exit(1)

DATASET_NAME = " + ".join([name for name, _ in datasets_loaded])
print(f"\n📊 Combined: {DATASET_NAME}")


✅ GPU: Tesla T4 | VRAM: 15.6 GB
   PyTorch 2.11.0+cu128 | Device: cuda

📋 Aether Text Emotion Model Config:
   Model: distilroberta-base
   Emotions: 15 categories
   Categories: ['happy', 'sad', 'angry', 'calm', 'anxious', 'energetic', 'focused', 'nostalgic', 'romantic', 'melancholic', 'confident', 'hopeful', 'frustrated', 'lonely', 'dreamy']
   Batch size: 32
   Epochs: 5
   Output: models/text_emotion

📥 Loading datasets (combining two sources for max coverage)...


✅ Loaded: 20-Emotion (2025)
   train: 79,595 samples
✅ Loaded: GoEmotions (Google)
   train: 43,410 samples
   validation: 5,426 samples
   test: 5,427 samples

📊 Combined: 20-Emotion + GoEmotions


In [ ]:
# ═══════════════════════════════════════════════
# SECTION 4: Map & Combine → 15 Aether Emotions
# ═══════════════════════════════════════════════

print("\n🔄 Mapping labels → 15 Aether emotions...")

GO_EMOTION_NAMES = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

all_texts = []
all_labels = []

for ds_name, ds in datasets_loaded:
    count_before = len(all_texts)

    if ds_name == "GoEmotions":
        go_to_aether = {}
        for idx, name in enumerate(GO_EMOTION_NAMES):
            aether = map_label_to_aether(name)
            if aether:
                go_to_aether[idx] = EMOTION_TO_ID[aether]

        split_name = "train" if "train" in ds else list(ds.keys())[0]
        for example in ds[split_name]:
            labels = example.get("labels", [])
            text = example.get("text", "")
            if isinstance(labels, list) and len(labels) > 0 and text:
                first = labels[0]
                if first in go_to_aether:
                    all_texts.append(text)
                    all_labels.append(go_to_aether[first])

    else:  # 20-Emotion
        split_name = list(ds.keys())[0]
        features = ds[split_name].features
        # Auto-detect label column name
        label_col = None
        for col in features:
            if col.lower() in ["label", "emotion", "labels", "sentiment"]:
                label_col = col
                break
        if label_col is None:
            label_col = [c for c in features if c.lower() not in ["text", "sentence", "content"]][0]

        # Auto-detect text column name
        text_col = None
        for col in features:
            if col.lower() in ["text", "sentence", "content", "utterance"]:
                text_col = col
                break
        if text_col is None:
            text_col = [c for c in features if c != label_col][0]

        print(f"   {ds_name} columns: text='{text_col}', label='{label_col}'")

        if hasattr(features[label_col], 'names'):
            original_labels = features[label_col].names
            label_mapping = {}
            unmapped = []
            for idx, name in enumerate(original_labels):
                aether = map_label_to_aether(name)
                if aether:
                    label_mapping[idx] = EMOTION_TO_ID[aether]
                else:
                    unmapped.append(name)
            if unmapped:
                print(f"   ⚠️  Unmapped from {ds_name}: {unmapped}")

            for example in ds[split_name]:
                orig = example[label_col]
                text = example.get(text_col, "")
                if orig in label_mapping and text:
                    all_texts.append(text)
                    all_labels.append(label_mapping[orig])
        else:
            for example in ds[split_name]:
                label = example[label_col]
                text = example.get(text_col, "")
                if isinstance(label, str) and text:
                    aether = map_label_to_aether(label)
                    if aether:
                        all_texts.append(text)
                        all_labels.append(EMOTION_TO_ID[aether])

    count_after = len(all_texts)
    print(f"   {ds_name} → {count_after - count_before:,} samples mapped")

print(f"\n   Total real-data samples: {len(all_texts):,}")

# ─── Check coverage ───
real_label_counts = Counter(all_labels)
covered = []
missing = []
for eid in range(NUM_LABELS):
    name = ID_TO_EMOTION[eid]
    if real_label_counts.get(eid, 0) > 0:
        covered.append(name)
    else:
        missing.append(name)

print(f"   Covered by real data: {len(covered)}/15 → {covered}")
if missing:
    print(f"   Missing (will augment): {missing}")

# ─── Data Augmentation for missing emotions ───
# Standard ML technique: curated seed examples for
# categories absent in source datasets. The pre-trained
# model already understands these concepts — the seeds
# teach label mapping, not language understanding.

AUGMENTATION_DATA = {
    "nostalgic": [
        "This song reminds me of those long summer days when I was a kid.",
        "I miss the way things used to be, those were simpler times.",
        "Looking at old photos always makes me feel something deep inside.",
        "I keep thinking about my childhood home and the memories we made there.",
        "That smell reminds me of my grandmother's kitchen, takes me right back.",
        "Sometimes I wish I could go back to those carefree college days.",
        "Hearing this melody brings back memories of road trips with friends.",
        "I found my old diary today and it made me emotional reading it.",
        "The neighborhood has changed so much, I barely recognize it anymore.",
        "I really miss the old days when we all used to hang out together.",
        "Watching that movie reminded me of my first date, feels like ages ago.",
        "I drove past my old school today and felt a wave of memories.",
        "These songs from 2010 bring back so many feelings and moments.",
        "I was cleaning and found our old family albums, couldn't stop looking.",
        "The rain today reminded me of monsoon evenings at my grandparents' house.",
        "I miss those late night conversations we used to have in hostel.",
        "Everything about this place reminds me of someone I used to know.",
        "Throwback to when life was simple and all we did was play outside.",
        "I still remember exactly how it felt the first time I heard this song.",
        "My heart aches for those times we can never get back.",
        "Old friendships that faded away still cross my mind sometimes.",
        "Walking through the old market felt like stepping back in time.",
        "I wish I had appreciated those moments more when I had them.",
        "The sunset today looked exactly like the ones from our summer trips.",
        "Some places just hold so many memories that they make you emotional.",
        "I keep going back to that one year that changed everything for me.",
        "Finding an old gift from a friend brought back a flood of emotions.",
        "I can still hear my mother's lullaby when I close my eyes at night.",
        "That coffee shop where we used to study is closed now, feels strange.",
        "Every time I visit my hometown, the memories come rushing back.",
    ],
    "melancholic": [
        "There's a deep heaviness in my chest that I can't seem to shake off.",
        "The world feels grey today, like all the color has been drained away.",
        "I feel like I'm carrying the weight of something I can't name.",
        "Everything feels pointless, like nothing really matters in the end.",
        "I sit by the window watching the rain and feel this deep emptiness.",
        "Life feels like it's passing me by and I'm just watching it happen.",
        "There's a quiet sadness in everything beautiful, knowing it won't last.",
        "Some days the world just feels heavier than I can bear.",
        "I feel like I'm drowning in my own thoughts and can't escape.",
        "The silence in this room is deafening and it fills me with sorrow.",
        "I look at people laughing and wonder why I can't feel that way.",
        "There's something deeply sorrowful about the way time keeps moving.",
        "I feel like a stranger in my own life, disconnected from everything.",
        "The beauty of the sunset tonight made me inexplicably sad.",
        "I've been staring at the ceiling for hours, lost in dark thoughts.",
        "Nothing feels real anymore, everything seems muted and distant.",
        "I feel hollow inside, like something important was taken from me.",
        "Even in a crowd, I feel this overwhelming sense of void.",
        "The music plays but it feels like it's echoing in an empty room.",
        "I wonder if this heaviness will ever lift or if it's just who I am now.",
        "Today everything feels like it's underwater, slow and muffled.",
        "I'm tired in a way that sleep can't fix, bone deep exhaustion.",
        "There's a persistent ache in my soul that nothing seems to soothe.",
        "I feel like I've lost something precious but I don't know what it is.",
        "The world keeps spinning but I feel like I've stopped moving.",
        "Some sorrows are too deep for tears, they just sit inside you.",
        "I watch the leaves fall and feel a kinship with them.",
        "There are moments where existence itself feels like a burden.",
        "I'm going through the motions but nothing feels meaningful.",
        "A deep, quiet despair has settled over me like fog.",
    ],
    "focused": [
        "I need to concentrate deeply on this problem, no distractions.",
        "I'm in the zone right now, completely absorbed in my work.",
        "Let me think about this carefully, I need to focus.",
        "I'm fully locked in on this task and nothing can break my attention.",
        "Time to put my head down and get this done with full concentration.",
        "I need complete silence so I can focus on this assignment.",
        "My mind is sharp right now, I can feel myself thinking clearly.",
        "I'm deeply engaged in studying, everything else has faded away.",
        "This requires my full undivided attention and careful analysis.",
        "I'm in a state of deep work, the world outside doesn't exist.",
        "I need to block out everything and concentrate on this code.",
        "My brain is firing on all cylinders, I'm solving this step by step.",
        "I can feel my mind working through this complex problem methodically.",
        "Right now all that matters is finishing this research paper.",
        "I've entered a flow state and I don't want to break it.",
        "Let me analyze this data carefully without any interruptions.",
        "I'm determined to crack this problem today, nothing else matters.",
        "I need laser focus right now, this deadline is critical.",
        "Every ounce of my mental energy is directed at this one task.",
        "I'm reading this research paper with intense concentration.",
        "My mind is completely clear and directed at solving this equation.",
        "I'm in deep study mode, please don't disturb me right now.",
        "This requires careful thought and I'm giving it my full attention.",
        "I can feel myself entering that productive mindset where work flows.",
        "Nothing exists right now except me and this project I'm building.",
        "I need to think through this architectural decision very carefully.",
        "My attention is razor sharp and I'm making real progress.",
        "I'm reviewing this code line by line with full concentration.",
        "I've silenced all notifications, it's pure focus time now.",
        "I'm dedicating the next few hours to deep, uninterrupted work.",
    ],
}

# Only augment for ACTUALLY missing emotions
augment_count = 0
for emotion, sentences in AUGMENTATION_DATA.items():
    if emotion in missing:
        eid = EMOTION_TO_ID[emotion]
        all_texts.extend(sentences)
        all_labels.extend([eid] * len(sentences))
        augment_count += len(sentences)
        print(f"   ✅ Augmented '{emotion}' with {len(sentences)} curated seed examples")

if augment_count > 0:
    pct = augment_count / len(all_texts) * 100
    print(f"\n   Augmentation total: {augment_count} samples ({pct:.1f}% of dataset)")

# ─── Build final train/validation split ───
import random
random.seed(42)
indices = list(range(len(all_texts)))
random.shuffle(indices)

split_idx = int(len(indices) * 0.85)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_ds = Dataset.from_dict({
    TEXT_COLUMN: [all_texts[i] for i in train_indices],
    "aether_label": [all_labels[i] for i in train_indices],
})
val_ds = Dataset.from_dict({
    TEXT_COLUMN: [all_texts[i] for i in val_indices],
    "aether_label": [all_labels[i] for i in val_indices],
})

mapped_dataset = DatasetDict({"train": train_ds, "validation": val_ds})
train_split = "train"

print(f"\n   📊 Final: Train {len(train_ds):,} | Validation {len(val_ds):,}")

# Show distribution
print("\n📊 Label distribution (training set):")
train_labels = mapped_dataset[train_split]["aether_label"]
label_counts = Counter(train_labels)
total = len(train_labels)
for eid in range(NUM_LABELS):
    count = label_counts.get(eid, 0)
    pct = count / total * 100 if total > 0 else 0
    bar = "█" * int(pct)
    emotion_name = ID_TO_EMOTION[eid]
    print(f"   {emotion_name:>12}: {count:>6} ({pct:>5.1f}%) {bar}")


# ═══════════════════════════════════════════════
# SECTION 5: Handle Class Imbalance
# ═══════════════════════════════════════════════
# Some emotions may have very few samples.
# We use class weights to handle this during training.

print("\n⚖️  Computing class weights for imbalanced data...")

from sklearn.utils.class_weight import compute_class_weight

train_labels_array = np.array(mapped_dataset[train_split]["aether_label"])
present_classes = np.unique(train_labels_array)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=train_labels_array,
)

# Create full weight tensor (15 classes)
weight_tensor = torch.ones(NUM_LABELS)
for cls, w in zip(present_classes, class_weights):
    weight_tensor[cls] = w
    print(f"   {ID_TO_EMOTION[cls]:>12}: weight = {w:.3f}")

weight_tensor = weight_tensor.to(DEVICE if DEVICE != "mps" else "cpu")
print("✅ Class weights computed — rare emotions get higher weight during training")


# ═══════════════════════════════════════════════
# SECTION 6: Tokenize
# ═══════════════════════════════════════════════

from transformers import AutoTokenizer

print(f"\n🔤 Tokenizing with {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples):
    tokenized = tokenizer(
        examples[TEXT_COLUMN],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    tokenized["labels"] = examples["aether_label"]
    return tokenized


# Get columns to remove
cols_to_remove = [c for c in mapped_dataset[train_split].column_names
                  if c not in ["input_ids", "attention_mask", "labels"]]

tokenized_dataset = mapped_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=cols_to_remove,
)
tokenized_dataset.set_format("torch")

print(f"✅ Tokenization complete | Max length: {MAX_LENGTH}")



🔄 Mapping labels → 15 Aether emotions...
   20-Emotion columns: text='sentence', label='emotion'
   20-Emotion → 79,595 samples mapped
   GoEmotions → 43,410 samples mapped

   Total real-data samples: 123,005
   Covered by real data: 12/15 → ['happy', 'sad', 'angry', 'calm', 'anxious', 'energetic', 'romantic', 'confident', 'hopeful', 'frustrated', 'lonely', 'dreamy']
   Missing (will augment): ['focused', 'nostalgic', 'melancholic']
   ✅ Augmented 'nostalgic' with 30 curated seed examples
   ✅ Augmented 'melancholic' with 30 curated seed examples
   ✅ Augmented 'focused' with 30 curated seed examples

   Augmentation total: 90 samples (0.1% of dataset)

   📊 Final: Train 104,630 | Validation 18,465

📊 Label distribution (training set):
          happy:  23836 ( 22.8%) ██████████████████████
            sad:  11587 ( 11.1%) ███████████
          angry:   8748 (  8.4%) ████████
           calm:  10837 ( 10.4%) ██████████
        anxious:  10257 (  9.8%) █████████
      energetic:   367

Map:   0%|          | 0/104630 [00:00<?, ? examples/s]

Map:   0%|          | 0/18465 [00:00<?, ? examples/s]

✅ Tokenization complete | Max length: 128


In [ ]:
# ═══════════════════════════════════════════════
# SECTION 7: Create Model with Weighted Loss
# ═══════════════════════════════════════════════

from transformers import AutoModelForSequenceClassification
import torch.nn as nn

print(f"\n🧠 Loading {MODEL_NAME}...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_EMOTION,
    label2id=EMOTION_TO_ID,
    problem_type="single_label_classification",
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded | {total_params:,} params ({trainable_params:,} trainable)")


# Custom Trainer with weighted loss for class imbalance
from transformers import Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    """Custom trainer that uses class weights for imbalanced emotions."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=weight_tensor.to(logits.device))
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


# ═══════════════════════════════════════════════
# SECTION 8: Training Setup
# ═══════════════════════════════════════════════

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("\n⚙️  Configuring training...")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels, predictions, average="weighted", zero_division=0),
    }


# Handle splits
available_splits = list(tokenized_dataset.keys())
train_key = "train" if "train" in available_splits else available_splits[0]

if "validation" in available_splits:
    eval_key = "validation"
elif "test" in available_splits:
    eval_key = "test"
elif len(available_splits) > 1:
    eval_key = available_splits[1]
else:
    split_data = tokenized_dataset[train_key].train_test_split(test_size=0.15, seed=42)
    tokenized_dataset = DatasetDict({
        "train": split_data["train"],
        "validation": split_data["test"],
    })
    train_key = "train"
    eval_key = "validation"

print(f"   Train: {train_key} ({len(tokenized_dataset[train_key]):,} samples)")
print(f"   Eval:  {eval_key} ({len(tokenized_dataset[eval_key]):,} samples)")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=(DEVICE == "cuda"),
    dataloader_num_workers=0 if DEVICE == "cuda" else 0,
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset[train_key],
    eval_dataset=tokenized_dataset[eval_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("✅ Training configured with weighted loss for class balance")


# ═══════════════════════════════════════════════
# SECTION 9: Train 🚀
# ═══════════════════════════════════════════════

print("\n" + "═" * 60)
print("🚀 TRAINING — Aether Text Emotion Model (15 emotions)")
print("═" * 60 + "\n")

train_result = trainer.train()

print("\n" + "═" * 60)
print("✅ TRAINING COMPLETE")
print("═" * 60)
print(f"   Time: {train_result.metrics['train_runtime']:.0f}s")
print(f"   Speed: {train_result.metrics['train_samples_per_second']:.1f} samples/sec")
print(f"   Loss: {train_result.metrics['train_loss']:.4f}")



🧠 Loading distilroberta-base...


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded | 82,129,935 params (82,129,935 trainable)

⚙️  Configuring training...
   Train: train (104,630 samples)
   Eval:  validation (18,465 samples)
✅ Training configured with weighted loss for class balance

════════════════════════════════════════════════════════════
🚀 TRAINING — Aether Text Emotion Model (15 emotions)
════════════════════════════════════════════════════════════



Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro,Precision,Recall
1,0.568202,0.677409,0.782616,0.782745,0.659661,0.791760,0.782616
2,0.504409,0.529767,0.800867,0.800621,0.819406,0.805862,0.800867
3,0.430709,0.517855,0.802004,0.799747,0.826558,0.807261,0.802004
4,0.343004,0.525156,0.814351,0.813043,0.801511,0.815360,0.814351
5,0.279657,0.528743,0.814406,0.813710,0.795121,0.816165,0.814406


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
✅ TRAINING COMPLETE
════════════════════════════════════════════════════════════
   Time: 1726s
   Speed: 303.1 samples/sec
   Loss: 0.5064


In [ ]:
# ═══════════════════════════════════════════════
# SECTION 10: Evaluate
# ═══════════════════════════════════════════════

from sklearn.metrics import classification_report, confusion_matrix

print("\n📈 Final Evaluation...")

eval_results = trainer.evaluate()
print(f"\n   Accuracy:        {eval_results['eval_accuracy']:.4f}")
print(f"   F1 (weighted):   {eval_results['eval_f1_weighted']:.4f}")
print(f"   F1 (macro):      {eval_results['eval_f1_macro']:.4f}")
print(f"   Precision:       {eval_results['eval_precision']:.4f}")
print(f"   Recall:          {eval_results['eval_recall']:.4f}")

# Classification report
predictions = trainer.predict(tokenized_dataset[eval_key])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Only include labels that appear in test set
present_in_test = sorted(set(labels))
target_names = [ID_TO_EMOTION[i] for i in present_in_test]

print("\n📋 Per-Emotion Classification Report:")
report = classification_report(
    labels, preds,
    labels=present_in_test,
    target_names=target_names,
    digits=4,
)
print(report)

# Save report
report_dict = classification_report(
    labels, preds,
    labels=present_in_test,
    target_names=target_names,
    digits=4,
    output_dict=True,
)


# ═══════════════════════════════════════════════
# SECTION 11: Save Model
# ═══════════════════════════════════════════════

print("\n💾 Saving model...")

final_path = OUTPUT_DIR / "final"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))

# Save comprehensive config
model_config = {
    "project": "Aether",
    "phase": "1A",
    "component": "Text Emotion Detection",
    "model_name": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "emotions": AETHER_EMOTIONS,
    "emotion_to_id": EMOTION_TO_ID,
    "id_to_emotion": {str(k): v for k, v in ID_TO_EMOTION.items()},
    "max_length": MAX_LENGTH,
    "dataset": DATASET_NAME,
    "training": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "weighted_loss": True,
    },
    "evaluation": {
        "accuracy": eval_results["eval_accuracy"],
        "f1_weighted": eval_results["eval_f1_weighted"],
        "f1_macro": eval_results["eval_f1_macro"],
        "precision": eval_results["eval_precision"],
        "recall": eval_results["eval_recall"],
        "per_emotion": report_dict,
    },
}

with open(OUTPUT_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print(f"✅ Model → {final_path}")
print(f"✅ Config → {OUTPUT_DIR / 'model_config.json'}")


# ═══════════════════════════════════════════════
# SECTION 12: Test Inference
# ═══════════════════════════════════════════════

print("\n" + "═" * 60)
print("🧪 LIVE INFERENCE TEST — 15 Emotion Predictions")
print("═" * 60)

from transformers import pipeline

emotion_pipeline = pipeline(
    "text-classification",
    model=str(final_path),
    tokenizer=str(final_path),
    top_k=NUM_LABELS,
    device=0 if DEVICE == "cuda" else -1,
)

# Test all 15 emotions
test_sentences = [
    # happy
    "I'm so thrilled, today has been absolutely wonderful!",
    # sad
    "Everything feels empty, I just want to cry.",
    # angry
    "I can't believe they betrayed me, I'm furious!",
    # calm
    "Just sitting quietly by the lake, feeling at peace.",
    # anxious
    "The exam is tomorrow and I haven't studied, I'm panicking.",
    # energetic
    "Let's go! I feel like I can run a marathon right now!",
    # focused
    "I need to concentrate deeply on solving this algorithm.",
    # nostalgic
    "This song reminds me of those childhood summer days.",
    # romantic
    "I can't stop thinking about her smile, my heart melts.",
    # melancholic
    "There's a deep heaviness in my soul that won't go away.",
    # confident
    "I know I'm going to crush this presentation, I'm ready.",
    # hopeful
    "Things are tough now but I believe tomorrow will be better.",
    # frustrated
    "This code keeps breaking and nothing makes sense anymore!",
    # lonely
    "Everyone seems to have someone, but I'm always alone.",
    # dreamy
    "I keep imagining floating through clouds of starlight.",
    # Hindi tests
    "मुझे बहुत खुशी हो रही है आज",           # happy
    "मैं बहुत चिंतित हूँ कल के बारे में",        # anxious
    "ये गाना मुझे पुराने दिनों की याद दिलाता है",  # nostalgic
]

print()
for sentence in test_sentences:
    results = emotion_pipeline(sentence)
    # Unwrap nested list (transformers v5 returns [[...]] for single input)
    if isinstance(results[0], list):
        results = results[0]
    top = results[0]
    top3 = results[:3]

    display = sentence[:55] + "..." if len(sentence) > 55 else sentence
    print(f"   📝 \"{display}\"")
    print(f"   → {top['label'].upper()} ({top['score']:.1%})", end="")
    print(f"  |  {top3[1]['label']} ({top3[1]['score']:.1%}), {top3[2]['label']} ({top3[2]['score']:.1%})")
    print()

# ═══════════════════════════════════════════════
# SECTION 13: Inference Utility Class
# ═══════════════════════════════════════════════

print("📦 Saving inference utility...")

inference_path = OUTPUT_DIR.parent.parent / "phase_1a_text_emotion" / "inference.py"
inference_path.parent.mkdir(parents=True, exist_ok=True)

# Write inference utility as clean lines (avoids escape issues)
inference_lines = [
    '"""',
    '═══════════════════════════════════════════════',
    'Aether — Text Emotion Inference Utility',
    'Loads the trained 15-emotion model and predicts',
    'emotions from text with intensity estimation.',
    '═══════════════════════════════════════════════',
    '"""',
    '',
    'import json',
    'import torch',
    'import numpy as np',
    'from pathlib import Path',
    'from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer',
    '',
    '',
    'class TextEmotionDetector:',
    '    """',
    '    Detects emotion from text input using Aether fine-tuned model.',
    '',
    '    Returns 15-emotion probability distribution with intensity levels.',
    '    Emotions: happy, sad, angry, calm, anxious, energetic, focused,',
    '              nostalgic, romantic, melancholic, confident, hopeful,',
    '              frustrated, lonely, dreamy',
    '    """',
    '',
    '    EMOTIONS = [',
    '        "happy", "sad", "angry", "calm", "anxious",',
    '        "energetic", "focused", "nostalgic", "romantic",',
    '        "melancholic", "confident", "hopeful", "frustrated",',
    '        "lonely", "dreamy",',
    '    ]',
    '',
    '    INTENSITY_THRESHOLDS = {',
    '        "neutral": (0.0, 0.15),',
    '        "mild": (0.15, 0.35),',
    '        "moderate": (0.35, 0.65),',
    '        "intense": (0.65, 1.0),',
    '    }',
    '',
    '    def __init__(self, model_path: str = None):',
    '        if model_path is None:',
    '            model_path = str(Path(__file__).parent.parent / "models" / "text_emotion" / "final")',
    '',
    '        self.model_path = model_path',
    '',
    '        # Load config',
    '        config_path = Path(model_path).parent / "model_config.json"',
    '        if config_path.exists():',
    '            with open(config_path) as f:',
    '                self.config = json.load(f)',
    '',
    '        # Device',
    '        device = 0 if torch.cuda.is_available() else -1',
    '',
    '        # Pipeline',
    '        self.pipe = pipeline(',
    '            "text-classification",',
    '            model=model_path,',
    '            tokenizer=model_path,',
    '            top_k=len(self.EMOTIONS),',
    '            device=device,',
    '        )',
    '',
    '    def _get_intensity(self, confidence: float) -> str:',
    '        """Map confidence score to intensity level."""',
    '        for level, (low, high) in self.INTENSITY_THRESHOLDS.items():',
    '            if low <= confidence < high:',
    '                return level',
    '        return "intense"',
    '',
    '    def predict(self, text: str) -> dict:',
    '        """',
    '        Predict emotion from text.',
    '',
    '        Returns:',
    '            {',
    '                "emotion": str,           # top predicted emotion',
    '                "confidence": float,      # 0-1 score',
    '                "intensity": str,         # neutral/mild/moderate/intense',
    '                "probabilities": dict,    # all 15 emotion scores',
    '                "top_3": list,            # top 3 emotions with scores',
    '            }',
    '        """',
    '        results = self.pipe(text)',
    '        probabilities = {r["label"]: round(r["score"], 4) for r in results}',
    '        top = results[0]',
    '        top_3 = [{"emotion": r["label"], "score": round(r["score"], 4)} for r in results[:3]]',
    '',
    '        return {',
    '            "emotion": top["label"],',
    '            "confidence": round(top["score"], 4),',
    '            "intensity": self._get_intensity(top["score"]),',
    '            "probabilities": probabilities,',
    '            "top_3": top_3,',
    '        }',
    '',
    '    def predict_batch(self, texts: list[str]) -> list[dict]:',
    '        """Predict emotions for a batch of texts."""',
    '        return [self.predict(t) for t in texts]',
    '',
    '    def get_emotion_vector(self, text: str) -> np.ndarray:',
    '        """',
    '        Returns emotion as a 15-dim vector for the fusion layer.',
    '        Order matches EMOTIONS list.',
    '        """',
    '        result = self.predict(text)',
    '        return np.array([result["probabilities"].get(e, 0.0) for e in self.EMOTIONS])',
    '',
    '',
    'if __name__ == "__main__":',
    '    detector = TextEmotionDetector()',
    '    tests = [',
    '        "I feel amazing today, everything is wonderful!",',
    '        "This reminds me of my childhood summers.",',
    '        "I cannot stop thinking about her.",',
    '    ]',
    '    for text in tests:',
    '        r = detector.predict(text)',
    '        emotion = r["emotion"].upper()',
    '        conf = r["confidence"]',
    '        intensity = r["intensity"]',
    '        print(f"Text: {text}")',
    '        print(f"  -> {emotion} | {conf:.1%} | Intensity: {intensity}")',
    '        print()',
]

with open(inference_path, "w") as f:
    f.write("\n".join(inference_lines) + "\n")

print(f"✅ Inference utility → {inference_path}")


# ═══════════════════════════════════════════════
# COMPLETE ✅
# ═══════════════════════════════════════════════

print("\n" + "═" * 60)
print("🎉 PHASE 1A COMPLETE — Aether Text Emotion Model")
print("═" * 60)
print(f"""
   ✅ 15 Emotion Categories (with intensity levels)
   ✅ Dataset: {DATASET_NAME}
   ✅ Model: {MODEL_NAME} (fine-tuned, weighted loss)
   ✅ Accuracy: {eval_results['eval_accuracy']:.4f}
   ✅ F1 (weighted): {eval_results['eval_f1_weighted']:.4f}
   ✅ F1 (macro): {eval_results['eval_f1_macro']:.4f}
   ✅ Model saved: {final_path}
   ✅ Inference utility: {inference_path}

   Next → Phase 1B: Voice Emotion Model (EN + HI)
""")



📈 Final Evaluation...


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted,F1 Macro,Precision,Recall
0.279657,0.528743,5,0.814406,0.813710,0.795121,0.816165,0.814406



   Accuracy:        0.8144
   F1 (weighted):   0.8137
   F1 (macro):      0.7951
   Precision:       0.8162
   Recall:          0.8144



📋 Per-Emotion Classification Report:
              precision    recall  f1-score   support

       happy     0.9109    0.8219    0.8641      4104
         sad     0.8815    0.8530    0.8670      2136
       angry     0.7479    0.8494    0.7954      1554
        calm     0.6484    0.5785    0.6115      1986
     anxious     0.8995    0.9237    0.9115      1744
   energetic     0.8429    0.8872    0.8645       665
     focused     0.7143    1.0000    0.8333         5
   nostalgic     0.6667    0.6667    0.6667         9
    romantic     0.7091    0.8515    0.7738       956
 melancholic     0.3333    1.0000    0.5000         1
   confident     0.9284    0.9663    0.9470       564
     hopeful     0.8314    0.8805    0.8552       728
  frustrated     0.6825    0.6685    0.6754      1650
      lonely     0.9374    0.9799    0.9582       596
      dreamy     0.7746    0.8342    0.8033      1767

    accuracy                         0.8144     18465
   macro avg     0.7672    0.8508    0.795

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model → models/text_emotion/final
✅ Config → models/text_emotion/model_config.json

════════════════════════════════════════════════════════════
🧪 LIVE INFERENCE TEST — 15 Emotion Predictions
════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



   📝 "I'm so thrilled, today has been absolutely wonderful!"
   → HAPPY (99.0%)  |  dreamy (0.5%), energetic (0.3%)

   📝 "Everything feels empty, I just want to cry."
   → SAD (77.9%)  |  lonely (20.8%), melancholic (0.8%)

   📝 "I can't believe they betrayed me, I'm furious!"
   → ANGRY (99.7%)  |  frustrated (0.2%), sad (0.0%)

   📝 "Just sitting quietly by the lake, feeling at peace."
   → HAPPY (99.0%)  |  lonely (0.5%), confident (0.1%)

   📝 "The exam is tomorrow and I haven't studied, I'm panicki..."
   → ANXIOUS (99.9%)  |  dreamy (0.0%), sad (0.0%)

   📝 "Let's go! I feel like I can run a marathon right now!"
   → HOPEFUL (83.7%)  |  confident (15.0%), focused (0.3%)

   📝 "I need to concentrate deeply on solving this algorithm."
   → FOCUSED (99.9%)  |  melancholic (0.0%), nostalgic (0.0%)

   📝 "This song reminds me of those childhood summer days."
   → NOSTALGIC (99.5%)  |  romantic (0.2%), focused (0.0%)

   📝 "I can't stop thinking about her smile, my heart melts."
   →

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Aether_models
!cp -r models/text_emotion /content/drive/MyDrive/Aether_models/
!cp phase_1a_text_emotion/inference.py /content/drive/MyDrive/Aether_models/
print("✅ Permanently saved to Google Drive!")


Mounted at /content/drive
✅ Permanently saved to Google Drive!
